In [2]:
import pandas as pd
import numpy as np
import seaborn.objects as so
from scipy import stats
import seaborn as sns

In [3]:
########## Question 1: Load data ##########
# * ~12% of rows have missing data in (year, school id, or student vol). 
# Mostly in student vol. We exclude these rows from our analysis
# * 0.3% of schools are missing data for certain years. 
# This does not alter the global picture, but can alter it at a country level

In [4]:
en = pd.read_csv("enrollments.csv", dtype={12: str, 13: str})

In [5]:
# en.head()

In [6]:
# en.describe()

In [7]:
# en.shape

In [8]:
en.iau_id1.unique().shape # number of universitiees

(17841,)

In [9]:
##### NULL values in key columns #####
# Removing rows with null years or null estimated students. 
# Drops ~12% of rows. Mostly null student volumes

In [10]:
{
    'student_vol': en.students5_estimated.isna().value_counts(),
    'year': en.year.isna().value_counts(),
    'school_id': en.iau_id1.isna().value_counts()
}

{'student_vol': students5_estimated
 False    142819
 True      18741
 Name: count, dtype: int64,
 'year': year
 False    161525
 True         35
 Name: count, dtype: int64,
 'school_id': iau_id1
 False    161553
 True          7
 Name: count, dtype: int64}

In [11]:
en_nona = en[(
    en.students5_estimated.isna() == False) & (
        en.year.isna() == False) & (en.iau_id1.isna() == False)].copy()

In [12]:
en_nona['iau_id1'].unique().shape

(16955,)

In [13]:
en_nona.shape

(142819, 27)

In [14]:
##### year coverage #####
# Do we have data from the start to the end of the school, 
# do we have data for all years in between? 
# A few 100 schools have missing years. This does not alter the global picture 
# (most student volume is from schools without misssing data). 
# It does impact the country level picture for some countires

In [15]:
yrs = en_nona.groupby(['countrycode', 'iau_id1', 'foundedyr', 'yrclosed'], dropna=False
                     ).agg({'year': ['min', 'max', 'count']})
# yrs

In [16]:
yrs.columns = ['min_yr', 'max_yr', 'count_yr']
yrs.reset_index(inplace=True)
yrs.min_yr.value_counts()

min_yr
1960.0    2297
1950.0    1855
2005.0    1759
2000.0    1693
1995.0    1555
2010.0    1481
1955.0     910
1990.0     902
1975.0     834
1965.0     773
1970.0     769
1985.0     700
2015.0     691
1980.0     632
2020.0     104
Name: count, dtype: int64

In [17]:
yrs['contain_start'] = np.where(
    yrs.min_yr - yrs.foundedyr <= 5, 'first data within 5yrs of founding', np.where(
        yrs.foundedyr > 1945, 'missing', 'founded 1945-'
    ))
yrs.contain_start.value_counts()

contain_start
first data within 5yrs of founding    12652
founded 1945-                          3587
missing                                 716
Name: count, dtype: int64

In [18]:
yrs['contain_end'] = np.where(
    yrs.max_yr == 2020, 'thru 2020', np.where(
        yrs.max_yr - yrs.yrclosed <= 5, 'thru closing', 'missing'
    ))
yrs.contain_end.value_counts()

contain_end
thru 2020       16683
missing           215
thru closing       57
Name: count, dtype: int64

In [19]:
yrs['est_count'] = (yrs.max_yr - yrs.min_yr)/5 + 1
yrs['miss_middle'] = np.where(yrs.est_count != yrs.count_yr, 1, 0)

In [20]:
yrs['miss_start'] = np.where(yrs['contain_start']== 'missing', 1, 0)
yrs['miss_end'] = np.where(yrs['contain_end']== 'missing', 1, 0)
yrs.groupby(['miss_end', 'miss_start', 'miss_middle']).count()
# most misses are from missing start of range, some from missing end, a few from missing middle

countrycode  iau_id1  foundedyr  yrclosed  \
miss_end miss_start miss_middle                                              
0        0          0                  15913    15953      15953        57   
                    1                     80       81         81         0   
         1          0                    667      704        704         5   
                    1                      2        2          2         0   
1        0          0                    204      205        205         0   
         1          0                     10       10         10         0   

                                 min_yr  max_yr  count_yr  contain_start  \
miss_end miss_start miss_middle                                            
0        0          0             15953   15953     15953          15953   
                    1                81      81        81             81   
         1          0               704     704       704            704   
                    1                 2       2         2              2   
1        0          0               205     205       205            205   
         1          0                10      10        10             10   

                                 contain_end  est_count  
miss_end miss_start miss_middle                          
0        0          0                  15953      15953  
                    1                     81         81  
         1          0                    704        704  
                    1                      2          2  
1        0          0                    205        205  
         1          0                     10         10

In [21]:
id_with_miss_yrs = yrs[(
    yrs.miss_start == 0) & (yrs.miss_end == 0) & (yrs.miss_middle)].iau_id1.to_list()

In [22]:
en_nona['miss_yrs'] = np.where(en_nona.iau_id1.isin(id_with_miss_yrs), 'yes', 'no')
vol_miss = en_nona.groupby(['year', 'miss_yrs']).agg({'students5_estimated':'sum'}).reset_index()

In [23]:
en_nona['miss_yrs'].value_counts() / en_nona.shape[0]

miss_yrs
no     0.99771
yes    0.00229
Name: count, dtype: float64

In [24]:
sns.lineplot(data=vol_miss[vol_miss['miss_yrs'] == 'yes'], x='year', y='students5_estimated')
plt.show()

NameError: name 'sns' is not defined

In [ ]:
sns.lineplot(data=vol_miss[vol_miss['miss_yrs'] == 'no'], x='year', y='students5_estimated')
plt.show()

In [ ]:
en_nona['num'] = (en_nona['miss_yrs'] == 'yes') * en_nona['students5_estimated']
y = en_nona.groupby('year').agg({'num': 'sum', 'students5_estimated': 'sum'}).reset_index()
y['pct_from_school_with_miss'] = y.num *1.0 / y.students5_estimated
y
# Schools with missing data do not drastically alter the global picture

In [ ]:
en_nona['num'] = (en_nona['miss_yrs'] == 'yes') * en_nona['students5_estimated']
y = en_nona.groupby(['year', 'country']).agg({'num': 'sum', 'students5_estimated': 'sum'}).reset_index()
y['pct_from_school_with_miss'] = y.num *1.0 / y.students5_estimated
y[y['pct_from_school_with_miss'] > 0.05].country.value_counts()

In [ ]:
########## Question 2: Detecting global events ##########
# Hypothesized detectable events: 
## Wars:
### Gulf: 1990-1991: Iraq, Kuwait, US, UK, France, Saudia Arabia, Egypt*
### Ukraine-Russia: 2015, 2022
## Introduction of birth control: 1960 with restrictions, 1970 more broadly, not until 1999 in Japan:
    # US, France, Japan, Germany, Canada
## Fall of the Berlin wall: 1989: Germany, Russia, Hungary*, Austria

# Findings

## Gulf war
### Kuwait student volume in the early 90s is decreased
### We do not see impact in: Iraq, US, UK, France, Saudia Arabia, or Egypt
## Ukraine-Russia: See a drop in student volumes in the late 2010s in both countries

## Birth control
### The 1960s are too close to the start of the data to detect an angle change
### 1960s also coincide with the baby boomers entering university so it would be hard to 
### associate anything to birth control. The boomer pattern is repeated every generation in the student 
### volume time series (France shows this pattern pretty clearly)

## Fall of the Berlin wall
### Germany and Russia show a small potential nudge up in student volume. 
### Hungary data (needs quality check). Student volume 3x in 20 years
### No impact is detectable in Austria

In [ ]:
country = 'Kuwait'
code = 'KWT'

In [ ]:
cc = 'austria'
en_nona[en_nona['country'].str.contains(str.lower(cc))].countrycode.value_counts()

In [ ]:
def make_country_ts(ccode, cname, id_with_miss_yrs):
    en_nona['exclude_ids_with_miss_yrs'] = np.where(
        en_nona.iau_id1.isin(id_with_miss_yrs), 0, en_nona.students5_estimated)
    c_yr = en_nona[en_nona['countrycode'] == ccode].groupby('year').agg(
        {'students5_estimated':'sum', 'exclude_ids_with_miss_yrs': 'sum' })
    return(
        so.Plot(c_yr, x="year")
        .add(so.Line(color="C1"), so.Agg(), y="students5_estimated", label='All available data')
        .add(so.Line(color="C2"), so.Agg(), y="exclude_ids_with_miss_yrs", label='Only universities without missing data')
        .label(x="Year", y="Student volume", title=cname + " student volume over time")
    )

In [ ]:
# Gulf war

In [ ]:
make_country_ts('KWT', 'Kuwait', id_with_miss_yrs)

In [ ]:
make_country_ts('IRQ', 'Iraq', id_with_miss_yrs)

In [ ]:
make_country_ts('USA', 'United States', id_with_miss_yrs)

In [ ]:
make_country_ts('GBR', 'United Kingdom', id_with_miss_yrs)

In [ ]:
make_country_ts('FRA', 'France', id_with_miss_yrs)

In [ ]:
make_country_ts('SAU', 'Saudia Arabia', id_with_miss_yrs)

In [ ]:
make_country_ts('EGY', 'Egypt', id_with_miss_yrs)

In [ ]:
# Ukraine and Russia

In [ ]:
make_country_ts('UKR', 'Ukraine', id_with_miss_yrs)

In [ ]:
make_country_ts('RUS', 'Russia', id_with_miss_yrs)

In [ ]:
# Birth control

In [ ]:
make_country_ts('USA', 'United States', id_with_miss_yrs)

In [ ]:
make_country_ts('FRA', 'France', id_with_miss_yrs)

In [ ]:
make_country_ts('JPN', 'Japan', id_with_miss_yrs)

In [ ]:
make_country_ts('DEU', 'Germany', id_with_miss_yrs)

In [ ]:
make_country_ts('CAN', 'Canada', id_with_miss_yrs)

In [ ]:
# Fall of the berlin wall

In [ ]:
make_country_ts('DEU', 'Germany', id_with_miss_yrs)

In [ ]:
make_country_ts('RUS', 'Russia', id_with_miss_yrs)

In [ ]:
make_country_ts('HUN', 'Hungary', id_with_miss_yrs)

In [ ]:
make_country_ts('AUT', 'Austria', id_with_miss_yrs)

In [ ]:
########## Question 3: GPD correlation ##########
# I looked at correlation of student growth (in 5yr increments) to GDP growth also in 5 year increments
# I also lagged GDP to see if current GDP affects current enrollment (people being able to afford university) vs.
# Educated individuals driving future GDP
# I excluded univerisities with incomplete data. 
# I also did some outlier removal I exclude GDP and student volume growth of more than 5x or falls 
# by more than 80% (with more time I would investigate more what is appropriate; look into what countries
# year get exclude)

# Findings

# GDP the year students are in university and 10 years prior matter for student volume growth. 
# Why a 10 year gap and not a 5 year gap I am not sure. I would want to plot these growth rates against 
# each other and look at whether certain years have a large effect

# GDP is most correlated to student volume growth in years prior. This suggests that growing higher ed, 
# grows GDP. With more time, I would validate this finding by looking at how it holds over year pairs.



In [ ]:
gdp = pd.read_csv("Countries GDP 1960-2020.csv")

In [ ]:
gdp_long = gdp.melt(id_vars=['Country Name', 'Country Code'])

In [ ]:
gdp_long['year'] = (np.ceil(gdp_long['variable'].astype(int)/5)*5).astype(int)
gdp5 = gdp_long.groupby(['Country Name', 'Country Code', 'year']).agg(
    {'value': 'mean'}).reset_index()

In [ ]:
def lag_gdp(gdp, lag):
    if lag / 5.0 != np.floor(lag/5.0):
        print ('You can only lag by 5 yr increments')
    else:
        gdp = gdp.sort_values(by=['Country Code', 'year']).reset_index(drop=True)
        gdp['prior'] = gdp.groupby('Country Code')['value'].shift(1)
        gdp['gdp_growth_lagged'] = (gdp['value'] / gdp['prior']).shift(int(lag / 5))
        return(gdp)

# lag_gdp(gdp5, 10)
# gdp5[['Country Name', 'Country Code', 'year', 'gdp_growth']]

In [ ]:
en_nona['exclude_ids_with_miss_yrs'] = np.where(
    en_nona.iau_id1.isin(id_with_miss_yrs), 0, en_nona.students5_estimated)
stu_yr_c = en_nona.groupby(['countrycode', 'year']).agg(
        {'students5_estimated':'sum', 'exclude_ids_with_miss_yrs': 'sum' }).reset_index()

stu_yr_c = stu_yr_c.sort_values(by=['countrycode', 'year']).reset_index(drop=True)
stu_yr_c['prior5yrs'] = stu_yr_c.groupby(
    'countrycode')['exclude_ids_with_miss_yrs'].shift(1)
stu_yr_c['stu_growth'] = stu_yr_c['exclude_ids_with_miss_yrs'] / stu_yr_c['prior5yrs']

In [ ]:
def get_corr(gdp, stu, lag, remove_outliers=5):
    gdp = lag_gdp(gdp, lag)
    df = pd.merge(
        gdp[['Country Name', 'Country Code', 'year', 'gdp_growth_lagged']], 
        stu[['countrycode', 'year', 'stu_growth']],
        left_on = ['Country Code', 'year'],
        right_on = ['countrycode', 'year']
    )
    df = df[(
        df.gdp_growth_lagged.isna() == False) & (df.stu_growth.isna() == False) & 
        (df.stu_growth < remove_outliers) & (df.gdp_growth_lagged < remove_outliers) & 
        (df.stu_growth > 1.0/remove_outliers) & (df.gdp_growth_lagged > 1.0/ remove_outliers)]
    return({
        'lag': 'GDP ' + str(lag) + " years ago",
        'n_obs': df.shape[0],
        'corr': stats.pearsonr(df['gdp_growth_lagged'], df['stu_growth'])})

In [ ]:
# for r in range(-30, 30, 5):
#     print(get_corr(gdp5, stu_yr_c, r, remove_outliers=1000)) # no outlier removal

In [ ]:
# for r in range(-30, 30, 5):
#     print(get_corr(gdp5, stu_yr_c, r, remove_outliers=2)) # no outlier removal

In [ ]:
for r in range(-30, 30, 5):
    print(get_corr(gdp5, stu_yr_c, r, remove_outliers=5)) # no outlier removal